## Sub-Phase 3.1 & Phase 5 Setup

In [1]:
# 0. Mount Google Drive for Persistent Storage
from google.colab import drive
import os
import sys

drive.mount('/content/drive')

# Create dedicated directories in your Drive
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Hiligaynon_NER_Project'
DATA_DIR = os.path.join(DRIVE_PROJECT_DIR, 'data')

os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, 'logs'), exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Google Drive mounted. Project data will be saved to: {DRIVE_PROJECT_DIR}")
print(f"ACTION REQUIRED: Ensure your .conll files are uploaded to: {DATA_DIR}")

Mounted at /content/drive
Google Drive mounted. Project data will be saved to: /content/drive/MyDrive/Hiligaynon_NER_Project
ACTION REQUIRED: Ensure your .conll files are uploaded to: /content/drive/MyDrive/Hiligaynon_NER_Project/data


In [2]:
# 1. Install Required Libraries
!pip install -q transformers datasets seqeval evaluate spacy sklearn-crfsuite
print("Dependencies installed.")

import numpy as np
import evaluate
from datasets import Dataset, DatasetDict

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.9 MB/s eta 0:00:00
Dependencies installed.


In [3]:
# 2. Model & Tokenizer Initialization
# (Ensure model_xlmr.py is uploaded to your Colab directory)
try:
    from model_xlmr import initialize_xlmr_model
except ModuleNotFoundError:
    print("WARNING: 'model_xlmr.py' not found! Make sure to upload it to the Colab runtime.")

# UPDATED: Full 29-label BIOES permutations for the 7 new target classes
label_list = [
    "O",
    "B-PERSON",   "I-PERSON",   "E-PERSON",   "S-PERSON",
    "B-ORG",      "I-ORG",      "E-ORG",      "S-ORG",
    "B-LOCATION", "I-LOCATION", "E-LOCATION", "S-LOCATION",
    "B-DATETIME", "I-DATETIME", "E-DATETIME", "S-DATETIME",
    "B-MONEY",    "I-MONEY",    "E-MONEY",    "S-MONEY",
    "B-EVENT",    "I-EVENT",    "E-EVENT",    "S-EVENT",
    "B-NORP",     "I-NORP",     "E-NORP",     "S-NORP"
]

print("Initializing Model...")
model, tokenizer, config = initialize_xlmr_model(label_list)

Initializing Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
# 3. Data Extraction (Parsing CoNLL to Hugging Face Datasets)
def read_conll(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"CRITICAL ERROR: Cannot find {file_path}. Please upload it to your Google Drive data folder.")

    with open(file_path, "r", encoding="utf-8") as f:
        sentences, labels, current_words, current_labels = [], [], [], []
        for line in f:
            if line.startswith("-DOCSTART-") or line == "" or line == "\n":
                if current_words:
                    sentences.append(current_words)
                    labels.append(current_labels)
                    current_words, current_labels = [], []
            else:
                # UPDATED: Dynamic split to handle both tabs and spaces safely
                splits = line.strip().split()
                if len(splits) >= 2:
                    current_words.append(splits[0])
                    current_labels.append(splits[1])
        if current_words:
            sentences.append(current_words)
            labels.append(current_labels)
    return {"tokens": sentences, "ner_tags": labels}

# Explicitly target the new 3-file structure
TRAIN_PATH = os.path.join(DATA_DIR, "dataset_train_final.conll")
VAL_PATH = os.path.join(DATA_DIR, "dataset_val_final.conll")
TEST_PATH = os.path.join(DATA_DIR, "dataset_test_final.conll")

print("Loading datasets...")
train_data = read_conll(TRAIN_PATH)
val_data = read_conll(VAL_PATH)
test_data = read_conll(TEST_PATH)

dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "validation": Dataset.from_dict(val_data),
    "test": Dataset.from_dict(test_data)     # <--- Test set is now officially registered
})

print(f"Loaded {len(dataset['train'])} train, {len(dataset['validation'])} validation, and {len(dataset['test'])} test samples.")

Loading datasets...
Loaded 5203 train, 648 validation, and 647 test samples.


## Sub-Phase 4.1: Environment Setup & Hyperparameter Tuning

In [5]:
# 4. Input Tokenization and Alignment
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # -100 tells PyTorch to ignore this token during loss calculation
            elif word_idx != previous_word_idx:
                # UPDATED: Strict tag checking to catch typos in the dataset early
                current_tag = label[word_idx]
                if current_tag not in config.label2id:
                    print(f"WARNING: Unrecognized tag '{current_tag}' found. Defaulting to 'O'.")
                label_ids.append(config.label2id.get(current_tag, 0))
            else:
                label_ids.append(-100) # Only label the first subword token
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Map the tokenizer across all 3 splits simultaneously
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

train_dataset = tokenized_datasets.get("train")
eval_dataset = tokenized_datasets.get("validation")  # <--- Trainer will use this for epoch evaluation
test_dataset = tokenized_datasets.get("test")        # <--- Held back for final evaluation

Map:   0%|          | 0/5203 [00:00<?, ? examples/s]

Map:   0%|          | 0/648 [00:00<?, ? examples/s]

Map:   0%|          | 0/647 [00:00<?, ? examples/s]

## Sub-Phase 4.2: Execution & Checkpointing

In [6]:
# 5. Entity-Level Evaluation Metrics
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [config.id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [config.id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

print("Pipeline Setup Complete! Ready to configure training.")

Pipeline Setup Complete! Ready to configure training.


In [7]:
import torch
from transformers import TrainingArguments, Trainer

# Set Hyperparameters based on constrained resources (Colab Pro T4/A100)
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# UPDATED: Increased to 8 epochs to handle the complex 29-label boundary distributions
EPOCHS = 8
WARMUP_RATIO = 0.1

training_args = TrainingArguments(
    output_dir=os.path.join(DRIVE_PROJECT_DIR, 'checkpoints'),
    logging_dir=os.path.join(DRIVE_PROJECT_DIR, 'logs'),
    logging_strategy='steps',
    logging_steps=50,
    eval_strategy='epoch', # Fixed deprecation warning here
    save_strategy='epoch',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    lr_scheduler_type='linear',
    warmup_ratio=WARMUP_RATIO,
    fp16=torch.cuda.is_available(),
)

print("TrainingArguments successfully initialized.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


TrainingArguments successfully initialized.


In [16]:
from transformers import DataCollatorForTokenClassification

def execute_training(model, tokenizer, train_dataset, eval_dataset, compute_metrics):
    data_collator = DataCollatorForTokenClassification(tokenizer)

    trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            processing_class=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics
        )

    print("Initiating training sequence...")
    print(f"Checkpoints will be saved to: {training_args.output_dir}")

    # Execute the core training loop
    trainer.train()

    final_save_path = os.path.join(DRIVE_PROJECT_DIR, 'checkpoints', 'best_model')
    trainer.save_model(final_save_path)

    print(f"Training finalized. Top-performing model safely preserved at {final_save_path}")
    return trainer

# EXECUTION TRIGGER (Uncommented and active)
trainer = execute_training(model, tokenizer, train_dataset, eval_dataset, compute_metrics)

Initiating training sequence...
Checkpoints will be saved to: /content/drive/MyDrive/Hiligaynon_NER_Project/checkpoints


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.470719,0.345088,0.541702,0.545377,0.543533,0.910126
2,0.242733,0.219949,0.708298,0.702290,0.705281,0.931704
3,0.196752,0.198715,0.740964,0.730280,0.735583,0.938198
4,0.162266,0.194152,0.721260,0.776930,0.748060,0.939036
5,0.141044,0.198772,0.722662,0.773537,0.747235,0.940433
6,0.109597,0.216512,0.686232,0.803223,0.740133,0.933101
7,0.091685,0.216048,0.722351,0.792197,0.755663,0.938338
8,0.086625,0.220785,0.715497,0.787108,0.749596,0.937779


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finalized. Top-performing model safely preserved at /content/drive/MyDrive/Hiligaynon_NER_Project/checkpoints/best_model
